# Guia Completo do Projeto
## Previsão diária de irradiância solar GHI com Machine Learning

Este notebook reúne o que é necessário para **entender, executar e explicar o projeto**:

- problema de pesquisa e objetivo da previsão;
- origem, período, granularidade e unidade dos dados;
- diferença entre média diária, média mensal e média móvel;
- limpeza, quantização e normalização;
- criação e alinhamento das features temporais;
- separação cronológica entre treino e teste;
- funcionamento dos modelos XGBoost e MLP;
- métricas, gráficos, resultados e artefatos;
- limitações, decisões metodológicas e perguntas comuns de banca.

> **Ideia central:** para cada localidade, o modelo recebe somente informações disponíveis até o dia `t` e estima o GHI do dia seguinte, `t+1`.

---

### Sumário

1. Problema, objetivo e escopo  
2. Conceitos de GHI, unidade e granularidade  
3. Dados e localidades  
4. Pipeline completo  
5. Coleta, validação e proveniência  
6. Média diária, mensal e móvel  
7. Limpeza e resolução diária  
8. Quantização e normalização  
9. Features temporais  
10. Alinhamento com o alvo e prevenção de vazamento  
11. Base final e divisão treino/teste  
12. Modelos  
13. Métricas e gráficos  
14. Resultados atuais  
15. Estrutura do código e execução  
16. Limitações e possíveis melhorias  
17. Roteiro de apresentação e perguntas de banca


# 1. Problema, objetivo e escopo

## Problema

A irradiância solar varia com estação do ano, latitude, nebulosidade e condições atmosféricas. Essa variação dificulta o planejamento de sistemas que dependem da disponibilidade de radiação solar.

## Objetivo geral

Desenvolver e avaliar um pipeline de Machine Learning capaz de prever o **GHI médio diário do próximo dia** em localidades associadas a fábricas de veículos elétricos.

## Formulação supervisionada

O problema é uma **regressão de série temporal**:

```text
Entradas: histórico do GHI disponível até o dia t
Saída:    GHI do dia seguinte, t+1
Horizonte de previsão: 1 dia à frente
```

Em notação simplificada:

```text
ŷ(t+1) = f(GHI(t), GHI(t-1), GHI(t-2), GHI(t-6), médias históricas)
```

## Escopo atual

- 10 localidades;
- dados diários de 1º de janeiro de 2019 a 31 de dezembro de 2024;
- 2 modelos: XGBoost e MLP;
- 7 features temporais;
- divisão 80%/20% cronológica;
- métricas MAE, MSE, RMSE e R².

O projeto prevê **irradiância**, não produção elétrica da fábrica, consumo energético ou potência de painéis fotovoltaicos. Para estimar energia fotovoltaica seriam necessários dados adicionais, como área dos módulos, eficiência, inclinação, temperatura e perdas do sistema.


# 2. O que é GHI?

**GHI** significa *Global Horizontal Irradiance*, ou Irradiância Global Horizontal. É a irradiância solar total recebida por uma superfície horizontal.

Conceitualmente, inclui:

```text
GHI = componente direta projetada no plano horizontal + componente difusa
```

## Unidade usada

Os dados da API são declarados em:

```text
W/m²
```

Essa unidade representa **potência por área em um instante ou intervalo médio**.

## Um cuidado importante

Neste projeto, o valor diário é a **média das observações horárias em W/m²**. Portanto:

- o resultado continua em `W/m²`;
- ele expressa a irradiância média do dia;
- ele não é uma soma de energia diária em `Wh/m²` ou `kWh/m²/dia`.

Se o objetivo fosse energia solar diária acumulada, seria necessário integrar a irradiância ao longo do tempo, e não simplesmente calcular sua média.


# 3. Dados e localidades

## Fonte

Os dados vêm do **NLR/NSRDB**:

- NLR: *National Laboratory of the Rockies*;
- NSRDB: *National Solar Radiation Database*;
- produto: `GOES Aggregated PSM v4`;
- variável solicitada: `ghi`;
- intervalo da API: 60 minutos;
- período oficial do projeto: 2019–2024.

## Localidades

| Localidade | País |
|---|---|
| BYD Camaçari | Brasil |
| Tesla Gigafactory Nevada | EUA |
| Tesla Gigafactory Texas | EUA |
| Hyundai Metaplant Georgia | EUA |
| Rivian Normal | EUA |
| Tesla Fremont Factory | EUA |
| Lucid AMP 1 Casa Grande | EUA |
| GM Factory Zero | EUA |
| Ford Rouge Electric Vehicle Center | EUA |
| BMW San Luis Potosí | México |

Cada localidade possui seu próprio CSV, sua própria preparação e seus próprios modelos. Os dados das dez localidades **não são misturados em um único treinamento**.

## Tamanho das bases

Cada CSV bruto validado contém:

```text
2.192 observações diárias
01/01/2019 a 31/12/2024
365 + 366 + 365 + 365 + 365 + 366 dias
```

Os anos bissextos de 2020 e 2024 estão incluídos.


# 4. Pipeline completo

```text
Cadastro auditável das localidades
            ↓
Coleta horária de GHI pela API NLR/NSRDB
            ↓
Agregação das observações para média diária
            ↓
Validação de origem, unidade, cobertura e integridade
            ↓
Limpeza e padronização da série
            ↓
Quantização do GHI em 128 níveis
            ↓
Normalização dos níveis para [0, 1]
            ↓
Criação de 4 lags + 3 médias móveis
            ↓
Criação do alvo do dia seguinte
            ↓
Remoção das linhas sem histórico/alvo
            ↓
Divisão cronológica: 80% treino / 20% teste
            ↓
Treinamento independente de XGBoost e MLP
            ↓
Previsões no período de teste
            ↓
MAE, MSE, RMSE, R², CSVs, modelos e gráficos
            ↓
Comparação dos modelos nas 10 localidades
```

## Regra metodológica central

Em qualquer linha usada pelo modelo:

```text
features ≤ dia t
alvo      = dia t+1
```

O futuro nunca pode entrar nas features.


# 5. Coleta, validação e proveniência

## Coleta

Para cada ano e localidade, a função `coletar_ghi_nrel` solicita à API:

```python
parameters=("ghi",)
time_step=60
leap_day=True
utc=False
```

Depois, as observações são agregadas por dia:

```python
daily = df_year[["ghi"]].resample("D").mean()
```

## Por que existem tantos metadados?

O CSV registra, além de `data` e `ghi`:

- nome, país, latitude e longitude da fábrica;
- endereço e fonte oficial da localidade;
- elemento OpenStreetMap e método de obtenção das coordenadas;
- fonte, produto, versão e endpoint da API;
- intervalo, agregação e unidade;
- ponto da grade NSRDB, identificação, fuso e elevação;
- data UTC da coleta.

Esses campos tornam a coleta **auditável e reproduzível**.

## Validações executadas

O script confirma:

- todas as colunas obrigatórias;
- fonte igual a `NLR/NSRDB`;
- produto e endpoint esperados;
- intervalo de 60 minutos e agregação `media_diaria`;
- unidade `W/m2`;
- cobertura diária completa de 2019 a 2024;
- ausência de datas inválidas ou duplicadas;
- GHI diário entre 0 e 500 W/m²;
- correspondência das coordenadas e da localidade;
- distância de até 5 km entre a fábrica e o ponto da grade NSRDB;
- presença no manifesto e igualdade do hash SHA-256.

Dados sintéticos não são usados como substituição quando a API falha.


# 6. Como funcionam as médias?

Esta é uma distinção essencial para explicar o projeto.

## 6.1 Média diária

A API fornece uma observação a cada 60 minutos. Para cada data, o projeto calcula:

```text
GHI_diário(d) = média das observações horárias do dia d
```

Exemplo simplificado:

```text
00h, 01h, ..., 23h → média → uma linha para aquele dia
```

Essa é a série usada pelo modelo.

## 6.2 Média mensal

No notebook de coleta, os valores diários são reamostrados no fim de cada mês:

```python
mensal = dados_diarios.resample("ME").mean()
```

Isso produz:

```text
média mensal = média dos valores diários pertencentes ao mês
```

**Uso:** somente visualização exploratória e comparação entre localidades.  
**Não é feature e não entra no treinamento.**

## 6.3 Média móvel

A média móvel é recalculada para cada dia usando uma janela deslizante:

```text
média móvel de 3 dias em 30/01
= média de 28/01, 29/01 e 30/01
```

No dia seguinte:

```text
média móvel de 3 dias em 31/01
= média de 29/01, 30/01 e 31/01
```

Ela não respeita o limite de um mês civil. Uma janela de 30 dias pode começar em dezembro e terminar em janeiro.

| Tipo | Origem | Uso |
|---|---|---|
| Média diária | observações horárias | forma a série principal |
| Média mensal | valores diários do mês | somente gráficos exploratórios |
| Média móvel | últimos 3, 7 ou 30 dias | entrada dos modelos |


# 7. Limpeza e resolução diária

A função `limpar_serie_ghi`:

1. detecta automaticamente a coluna de data;
2. detecta a coluna de GHI;
3. converte datas para `datetime`;
4. converte o GHI para número;
5. remove datas e valores inválidos;
6. remove GHI negativo, por não ter interpretação física neste contexto;
7. ordena cronologicamente;
8. remove datas duplicadas, mantendo o último registro;
9. garante frequência diária com `resample("D").mean()`.

## Por que agregar novamente se os CSVs já são diários?

A função também aceita CSV, Excel ou Parquet fornecidos pelo usuário. Uma entrada pode ser horária, subdiária ou possuir mais de uma observação por dia. A etapa garante que o restante do pipeline sempre receba o mesmo contrato:

```text
data       ghi
2019-01-01 valor diário
2019-01-02 valor diário
...
```

## Dias ausentes

O `resample` cria a grade diária, e dias sem nenhuma observação ficam ausentes e são removidos. Nos dez CSVs oficiais, uma validação anterior exige cobertura diária completa, portanto não há lacunas no período 2019–2024.


# 8. Quantização e normalização

## 8.1 Quantização em 128 níveis

O GHI contínuo é mapeado para inteiros de 0 a 127:

```text
0, 1, 2, ..., 127
```

Fórmula conceitual:

```text
q = arredondar((x - mínimo) / (máximo - mínimo) × 127)
```

Valores fora da faixa ajustada são limitados aos extremos.

### Objetivo

- representar o sinal em uma escala discreta controlada;
- reduzir pequenas variações do valor contínuo;
- manter uma representação comum para os dois modelos.

### Consequência

A quantização perde parte da precisão original. Ela é uma decisão metodológica do projeto, não uma exigência de XGBoost ou MLP.

## 8.2 Normalização

Depois da quantização:

```text
ghi_normalizado = ghi_quantizado / 127
```

O resultado fica no intervalo `[0, 1]`. Isso é especialmente importante para o MLP, cujo treinamento é sensível à escala.

## 8.3 Proteção contra vazamento

O mínimo e o máximo da quantização são ajustados usando somente o trecho pertencente ao treinamento, incluindo seus alvos. O conjunto de teste é transformado com esses mesmos parâmetros.

Assim, estatísticas do futuro não são usadas para definir a escala do passado.


# 9. Features temporais

O modelo recebe sete colunas:

| Feature | Significado em relação ao alvo `t+1` |
|---|---|
| `ghi_t-1` | GHI normalizado do dia `t` |
| `ghi_t-2` | GHI normalizado do dia `t-1` |
| `ghi_t-3` | GHI normalizado do dia `t-2` |
| `ghi_t-7` | GHI normalizado do dia `t-6` |
| `ghi_media_movel_3d` | média normalizada dos dias `t-2` a `t` |
| `ghi_media_movel_7d` | média normalizada dos dias `t-6` a `t` |
| `ghi_media_movel_30d` | média normalizada dos dias `t-29` a `t` |

## Atenção ao nome dos lags

No código, a linha de data `t` prevê `t+1`. Por isso:

```python
dados["ghi_t-1"] = dados["ghi_normalizado"].shift(0)
```

`ghi_t-1` significa **um dia antes do alvo**, não um dia antes da data da linha.

Os deslocamentos usados são:

```python
ghi_t-1 → shift(0)
ghi_t-2 → shift(1)
ghi_t-3 → shift(2)
ghi_t-7 → shift(6)
```

## Por que usar esses horizontes?

- 1, 2 e 3 dias: persistência e comportamento recente;
- 7 dias: referência de uma semana;
- média de 3 dias: nível recente;
- média de 7 dias: suavização semanal;
- média de 30 dias: tendência mais lenta e sazonalidade aproximada.

As médias móveis são calculadas sobre o **GHI quantizado e normalizado**, não diretamente sobre o valor original em W/m².


# 10. Alinhamento entre features e alvo

Considere valores normalizados fictícios:

| Data | GHI normalizado |
|---|---:|
| 01/01 | 0,20 |
| 02/01 | 0,30 |
| 03/01 | 0,40 |
| 04/01 | 0,50 |

Para a linha de 03/01:

```text
data da linha = 03/01
ghi_t-1      = 0,40  (03/01)
ghi_t-2      = 0,30  (02/01)
ghi_t-3      = 0,20  (01/01)
média 3d     = (0,20 + 0,30 + 0,40) / 3 = 0,30
data_alvo    = 04/01
ghi_alvo     = 0,50
```

O código cria o alvo com:

```python
dados["data_alvo"] = dados["data"].shift(-1)
dados["ghi_alvo"] = dados["ghi_normalizado"].shift(-1)
```

## Por que não há vazamento temporal?

- as janelas terminam no dia `t`;
- o alvo está no dia `t+1`;
- a divisão treino/teste preserva a ordem;
- a escala é ajustada no treino;
- o projeto possui teste automatizado que verifica esse alinhamento.

Uma forma errada seria deslocar a série para o futuro antes de calcular a média, pois isso permitiria que o valor a prever entrasse nas entradas.


# 11. Base final de modelagem

Principais colunas do arquivo `*_features.csv`:

| Grupo | Colunas |
|---|---|
| Referência | `data`, `ghi` |
| Transformações | `ghi_quantizado`, `ghi_normalizado` |
| Entradas | `ghi_t-1`, `ghi_t-2`, `ghi_t-3`, `ghi_t-7` |
| Entradas | `ghi_media_movel_3d`, `ghi_media_movel_7d`, `ghi_media_movel_30d` |
| Alvo | `data_alvo`, `ghi_alvo` |
| Auditoria do alvo | `ghi_alvo_quantizado`, `ghi_alvo_original` |

## Por que a primeira linha é 30/01/2019?

A maior janela precisa de 30 observações:

```text
01/01 a 30/01 → primeiro histórico completo de 30 dias
```

Também é necessário existir o dia seguinte como alvo. Por isso a última linha de entrada é 30/12/2024, cujo alvo é 31/12/2024.

## Contagem

```text
Base diária original:             2.192 linhas
Perda inicial pela janela de 30d:    29 linhas
Perda final pela ausência de t+1:     1 linha
Base de modelagem:                2.162 linhas
```

Essa remoção é esperada e não representa perda acidental de dados.


# 12. Divisão cronológica treino/teste

Depois da criação das features:

```text
80% inicial → treino
20% final   → teste
```

Para cada localidade:

```text
Total:  2.162 exemplos
Treino: 1.729 exemplos
Teste:    433 exemplos
```

Períodos dos alvos:

```text
Treino: 31/01/2019 a 25/10/2023
Teste:  26/10/2023 a 31/12/2024
```

## Por que não embaralhar?

Em uma aplicação real, o modelo é treinado com o passado e utilizado no futuro. Embaralhar poderia colocar observações futuras no treino e observações passadas no teste, produzindo uma avaliação artificialmente otimista.

## O que o teste representa?

O conjunto final simula uma utilização posterior ao período usado para ajuste. As métricas mostram o desempenho em dados cronologicamente mais novos e não vistos durante o treinamento.


# 13. Modelos

Os dois modelos usam exatamente as mesmas sete features e o mesmo alvo. Isso permite uma comparação coerente.

## 13.1 XGBoost

O `XGBRegressor` combina muitas árvores de decisão construídas sequencialmente. Cada nova árvore busca reduzir os erros das anteriores.

Hiperparâmetros:

```python
n_estimators=300
max_depth=3
learning_rate=0.05
subsample=0.9
colsample_bytree=0.9
objective="reg:squarederror"
random_state=42
n_jobs=-1
```

### Pontos fortes

- bom desempenho em dados tabulares;
- captura relações não lineares;
- pouca exigência de escala;
- permite estudar importância das features.

## 13.2 MLP

O `MLPRegressor` é uma rede neural totalmente conectada:

```text
7 entradas → 64 neurônios → 32 neurônios → 1 saída
```

Hiperparâmetros:

```python
hidden_layer_sizes=(64, 32)
activation="relu"
solver="adam"
max_iter=1000
learning_rate_init=0.001
random_state=42
```

### Pontos fortes

- aprende relações não lineares;
- combina as features em representações internas;
- beneficia-se da normalização `[0, 1]`.

## Pós-processamento

As previsões são limitadas ao intervalo válido:

```python
predicao.clip(0, 1)
```

Isso impede saídas normalizadas negativas ou superiores a 1.


# 14. Métricas de avaliação

As métricas são calculadas sobre o **alvo quantizado e normalizado**, portanto MAE e RMSE estão na escala `[0, 1]`.

## MAE — Erro Absoluto Médio

```text
MAE = média(|real - previsto|)
```

- fácil de interpretar;
- todos os erros têm peso linear;
- menor é melhor.

## MSE — Erro Quadrático Médio

```text
MSE = média((real - previsto)²)
```

- penaliza mais fortemente erros grandes;
- menor é melhor.

## RMSE — Raiz do Erro Quadrático Médio

```text
RMSE = √MSE
```

- volta à mesma escala normalizada do alvo;
- sensível a erros grandes;
- menor é melhor.

## R² — Coeficiente de determinação

```text
R² = 1 - erro do modelo / variabilidade total
```

Interpretação:

| R² | Leitura |
|---:|---|
| próximo de 1 | modelo explica grande parte da variação |
| próximo de 0 | desempenho próximo de prever a média |
| negativo | pior que a referência da média |

R² maior é melhor, mas deve ser analisado junto com MAE, RMSE e gráficos.

## Conversão aproximada do erro

Como houve quantização baseada no mínimo e máximo do treino, um erro normalizado só pode ser convertido para W/m² conhecendo os parâmetros daquela localidade:

```text
erro aproximado em W/m² ≈ erro normalizado × (máximo_treino - mínimo_treino)
```

Devido ao arredondamento da quantização, essa conversão é aproximada.


# 15. Gráficos e artefatos

## Gráficos gerados por localidade

1. série temporal de teste com real, XGBoost e MLP;
2. real versus previsto do XGBoost;
3. dispersão real versus previsto do XGBoost;
4. real versus previsto da MLP;
5. dispersão real versus previsto da MLP.

### Como interpretar

- linhas próximas no gráfico temporal indicam bom acompanhamento da dinâmica;
- na dispersão, pontos próximos da diagonal representam previsões próximas do real;
- afastamentos sistemáticos podem indicar viés;
- dificuldade em picos e quedas mostra limitações diante de mudanças rápidas.

## Arquivos gerados

```text
dados/processados/localidades_ev/*_features.csv
resultados/modelos/localidades/*.joblib
resultados/todas_localidades/metricas_geral.csv
resultados/todas_localidades/resumo_localidades.csv
resultados/todas_localidades/previsoes/<localidade>/*.csv
resultados/todas_localidades/figuras/<localidade>/*.png
dados/brutos/localidades_ev/manifesto_nsrdb.csv
```

Os arquivos `.joblib` guardam os modelos treinados. Para fazer previsões futuras de forma reproduzível também é necessário conservar os parâmetros de quantização/normalização e construir as mesmas sete features.


# 16. Resultados atuais

Resumo dos resultados já gerados:

| Localidade | Melhor modelo | Melhor R² |
|---|---|---:|
| BYD Camaçari | MLP | 0,3884 |
| Tesla Gigafactory Nevada | XGBoost | 0,8596 |
| Tesla Gigafactory Texas | MLP | 0,5716 |
| Hyundai Metaplant Georgia | MLP | 0,5190 |
| Rivian Normal | MLP | 0,6401 |
| Tesla Fremont Factory | XGBoost | 0,8698 |
| Lucid AMP 1 Casa Grande | XGBoost | 0,8167 |
| GM Factory Zero | MLP | 0,6366 |
| Ford Rouge Electric Vehicle Center | XGBoost | 0,6512 |
| BMW San Luis Potosí | MLP | 0,5903 |

## Síntese

```text
MLP foi melhor em R²:      6 localidades
XGBoost foi melhor em R²:  4 localidades
```

Médias entre as dez localidades:

| Modelo | MAE médio | RMSE médio | R² médio |
|---|---:|---:|---:|
| XGBoost | 0,1068 | 0,1410 | 0,6398 |
| MLP | 0,1052 | 0,1390 | 0,6529 |

## Leitura correta

- nenhum modelo vence em todas as localidades;
- a MLP possui resultado médio ligeiramente melhor;
- XGBoost alcança os maiores R² individuais;
- diferenças climáticas e de previsibilidade entre localidades afetam o desempenho;
- resultados baixos, como em Camaçari, indicam maior dificuldade para explicar variações diárias usando somente o histórico do próprio GHI.

Não se deve concluir causalidade entre a fábrica e a irradiância. As fábricas funcionam como pontos geográficos de estudo.


# 17. Estrutura do projeto

| Arquivo ou pasta | Responsabilidade |
|---|---|
| `codigo_fonte/preprocessamento.py` | coleta, limpeza, agregação diária, quantização e normalização |
| `codigo_fonte/features.py` | lags, médias móveis, alvo e divisão temporal |
| `codigo_fonte/modelos.py` | treinamento e salvamento de XGBoost e MLP |
| `codigo_fonte/avaliacao.py` | métricas e CSVs de previsão |
| `codigo_fonte/graficos.py` | gráficos temporais e de dispersão |
| `codigo_fonte/localidades_ev.py` | cadastro auditável das fábricas |
| `treinamento_principal.py` | pipeline para uma série |
| `treinar_todas_localidades.py` | coleta, validação e treino nas 10 localidades |
| `cadernos_jupyter/00_*` | coleta, validação e visualização dos dados |
| `cadernos_jupyter/01_*` | explicação completa do método |
| `cadernos_jupyter/02_*` | análise dos resultados |
| `testes/` | testes automatizados |

## Separação de responsabilidades

Os notebooks documentam e apresentam. A lógica reutilizável permanece nos módulos Python. Isso reduz duplicação, facilita testes e permite executar o pipeline sem depender da interface do Jupyter.


# 18. Como executar

## Instalar dependências

```bash
pip install -r requirements.txt
```

## Validar os dados oficiais

```bash
python treinar_todas_localidades.py --validar-dados
```

## Baixar novamente os dados

Requer `NREL_API_KEY` e `NREL_EMAIL` no arquivo `.env`:

```bash
python treinar_todas_localidades.py --somente-download --forcar-download
```

## Treinar as dez localidades

```bash
python treinar_todas_localidades.py
```

## Treinar uma única série

```bash
python treinamento_principal.py \
  --data-path dados/brutos/localidades_ev/byd_camacari.csv
```

## Executar testes

```bash
pytest
```

## Exportar os notebooks para HTML

```bash
jupyter nbconvert --to html \
  cadernos_jupyter/01_explicacao_teorica_pipeline.ipynb \
  --output-dir relatorios
```


# 19. Limitações e melhorias possíveis

## Limitações atuais

1. **Somente o histórico do GHI é usado.** Não entram nuvens, temperatura, umidade, precipitação ou previsão meteorológica.
2. **Horizonte único.** O modelo prevê apenas um dia à frente.
3. **Divisão única 80/20.** Não há validação temporal em múltiplas janelas.
4. **Hiperparâmetros fixos.** Não foi realizada busca sistemática por configuração.
5. **Quantização perde resolução.** É necessário comparar com treinamento diretamente no GHI contínuo.
6. **Média diária inclui horas noturnas.** Isso é coerente com o valor médio de 24 horas, mas outra definição poderia usar apenas período diurno ou energia integrada.
7. **Features sazonais explícitas não são usadas.** Mês, dia do ano, seno/cosseno sazonais e posição solar poderiam ajudar.
8. **Não existe baseline explícito nas tabelas.** Comparar com persistência, como `previsão = GHI de hoje`, fortaleceria a avaliação.
9. **Métricas principais estão normalizadas.** Relatar também erros em W/m² facilitaria a interpretação física.
10. **Modelos são locais.** Não existe um modelo global que aprenda conjuntamente com latitude, longitude e localidade.

## Melhorias prioritárias

- adicionar baseline de persistência;
- incluir variáveis meteorológicas;
- usar validação *walk-forward*;
- comparar série contínua contra série quantizada;
- adicionar features sazonais;
- otimizar hiperparâmetros somente dentro do treino;
- salvar parâmetros de transformação junto ao modelo;
- calcular métricas também na escala original.

Esses pontos não invalidam o trabalho. Eles delimitam o que foi avaliado e indicam continuidade científica.


# 20. Perguntas comuns de banca

## “A média usada é diária ou mensal?”

O modelo usa uma **série diária**. Cada dia é a média das observações horárias. A média mensal aparece apenas em gráficos exploratórios. As features incluem médias **móveis** de 3, 7 e 30 dias.

## “Por que a unidade continua W/m²?”

Porque foi calculada a média temporal da irradiância, não sua integral. Uma média de valores em W/m² continua em W/m².

## “Por que usar média de 30 dias e não média do mês?”

A janela móvel acompanha continuamente os 30 dias mais recentes. Ela evita mudanças bruscas na virada do mês e está disponível para qualquer data.

## “O modelo vê o valor do dia que deve prever?”

Não. A linha do dia `t` usa dados até `t` e prevê `t+1`. Esse alinhamento é implementado com `shift` e verificado por teste automatizado.

## “Por que não embaralhar os dados?”

Porque a ordem temporal representa a aplicação real. Embaralhar permitiria que o futuro influenciasse o treinamento.

## “Por que quantizar?”

Para representar o sinal em 128 níveis e reduzir pequenas variações. Porém, isso também perde precisão e deve ser tratado como escolha experimental.

## “Qual modelo foi melhor?”

Depende da localidade. A MLP venceu em seis e o XGBoost em quatro pelo R². Em média, a MLP foi ligeiramente melhor, mas o XGBoost obteve os maiores R² individuais.

## “É previsão do clima ou de geração solar?”

É previsão do GHI médio diário. Não é previsão meteorológica completa nem cálculo direto de energia fotovoltaica.

## “Por que escolher fábricas de veículos elétricos?”

Elas definem locais geográficos relevantes para comparar disponibilidade solar. O projeto não afirma que as fábricas causam mudanças na irradiância.

## “O resultado serve para operação em tempo real?”

Como prova de conceito preditiva, sim. Para implantação operacional ainda seriam necessários atualização automática dos dados, persistência dos transformadores, monitoramento e variáveis meteorológicas previstas.


# 21. Roteiro curto para apresentar o projeto

> “O trabalho avalia a previsão diária de irradiância solar em dez localidades de fábricas de veículos elétricos. Os dados vêm da base oficial NLR/NSRDB, originalmente com intervalo de 60 minutos, e são agregados em uma média para cada dia entre 2019 e 2024.
>
> Depois da limpeza, o GHI é quantizado em 128 níveis e normalizado entre zero e um. Para transformar a série temporal em um problema supervisionado, criamos quatro defasagens e três médias móveis, totalizando sete features. Cada linha usa informações disponíveis até o dia atual para prever o GHI do dia seguinte.
>
> A separação é cronológica: os primeiros 80% dos exemplos são usados no treinamento e os 20% finais no teste. Comparamos XGBoost e uma rede neural MLP usando MAE, MSE, RMSE e R².
>
> Os resultados mostram que não existe um único vencedor em todas as localidades: a MLP foi melhor em seis e o XGBoost em quatro. Isso reforça que a previsibilidade varia conforme o local. Como limitações, o modelo usa apenas o histórico do GHI e ainda não inclui variáveis meteorológicas ou validação temporal em múltiplas janelas.” 

## Frase central para memorizar

```text
Dados horários → média diária → transformações → histórico temporal
→ previsão do dia seguinte → avaliação cronológica
```


# 22. Resumo final

| Pergunta | Resposta |
|---|---|
| O que é previsto? | GHI médio diário do dia seguinte |
| Qual é a fonte? | NLR/NSRDB, GOES Aggregated PSM v4 |
| Qual período? | 2019–2024 |
| Qual granularidade do modelo? | diária |
| A média mensal entra no modelo? | não, somente na visualização |
| Quais features? | lags 1, 2, 3 e 7; médias móveis 3, 7 e 30 dias |
| Qual alvo? | GHI quantizado e normalizado de `t+1` |
| Quantos exemplos por local? | 2.162 |
| Como é a divisão? | 1.729 treino e 433 teste, em ordem temporal |
| Quais modelos? | XGBoost e MLP |
| Quais métricas? | MAE, MSE, RMSE e R² |
| Como se evita vazamento? | escala no treino, features até `t`, alvo em `t+1` e divisão cronológica |

O ponto mais importante é compreender que o projeto mantém a causalidade temporal: **o passado e o presente formam as entradas; o próximo dia é o alvo**.
